[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# Merging &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import math
from datetime import datetime, timedelta

import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})

print("pandas", pd.__version__, "|", len(readings), "readings,", len(stations), "stations")

pandas 2.2.3 | 35040 readings, 5 stations


**1.** Merge `means` and `stations` so that every station appears, Kirkenes included, with `means` as the left table.

In [2]:
means = readings.groupby("station")["celsius"].mean().round(2).reset_index(name="mean")

pd.merge(means, stations, on="station", how="right")

,station,mean,latitude
0,Bergen,8.00,60.39
1,Oslo,6.50,59.91
2,Svalbard,-4.48,78.22
3,Tromso,3.50,69.65
4,Kirkenes,NaN,69.73


With `means` on the left, Kirkenes is only in the right table, so a right join keeps it, as an outer join would.

**2.** Using `located`, print the mean reading of the stations south of 61 degrees.

In [3]:
located = pd.merge(readings, stations, on="station", how="left", validate="many_to_one")
south = located[located["latitude"] < 61]

print(south["station"].unique().tolist())
print(round(south["celsius"].mean(), 2))

['Bergen', 'Oslo']
7.25


Bergen and Oslo are the stations south of 61 degrees, and their readings average to the mean of the two stations' means, since each has a reading for every hour.

**3.** Make an outer merge of `mainland` and `means` with `indicator=True`, and show only the rows that came from one table alone.

In [4]:
means = readings.groupby("station")["celsius"].mean().round(2).reset_index(name="mean")
mainland = stations[stations["station"] != "Svalbard"]
matched = pd.merge(mainland, means, on="station", how="outer", indicator=True)

matched[matched["_merge"] != "both"]

,station,latitude,mean,_merge
1,Kirkenes,69.73,NaN,left_only
3,Svalbard,NaN,-4.48,right_only


Kirkenes is `left_only` and Svalbard is `right_only`.

**4.** Make a table of each station's coldest and warmest reading of the year, from two groupby results merged on `station` with suffixes `_coldest` and `_warmest`.

In [5]:
coldest = readings.groupby("station")["celsius"].min().reset_index()
warmest = readings.groupby("station")["celsius"].max().reset_index()

pd.merge(coldest, warmest, on="station", suffixes=("_coldest", "_warmest"))

,station,celsius_coldest,celsius_warmest
0,Bergen,-4.8,20.8
1,Oslo,-6.3,19.3
2,Svalbard,-17.3,8.3
3,Tromso,-9.2,16.2


Both results call their column `celsius`, so the suffixes tell them apart.

**5.** Stack January's and July's means with `concat`, with a `month` column, and print the station and month of the lowest mean.

In [6]:
month = readings["time"].dt.month
january = readings[month == 1].groupby("station")["celsius"].mean().round(1).reset_index()
july = readings[month == 7].groupby("station")["celsius"].mean().round(1).reset_index()

both = pd.concat([january.assign(month="January"), july.assign(month="July")], ignore_index=True)
lowest = both.loc[both["celsius"].idxmin()]
print(lowest["station"], lowest["month"], lowest["celsius"])

Svalbard January -13.4


`idxmin` returns the label of the lowest mean, and `ignore_index=True` made the labels unique, so `loc` returns one row. Svalbard in January is the lowest.

---

&#8592; **Back to:** [Merging](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/15-merging.ipynb)  &nbsp;&middot;&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)
